# [9665] Locality-Sensitive Hashing 1
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Amazon_prime_titles.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/17/25 12:34:54


### Import libraries

In [ ]:
%%time

! pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 1.5 MB/s eta 0:00:00
CPU times: user 95.5 ms, sys: 21.6 ms, total: 117 ms
Wall time: 11.2 s


In [ ]:
import numpy as np
import pandas as pd
import time
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from datasketch import MinHash
from datasketch import MinHashLSHForest

In [ ]:
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Load data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/Amazon_prime_titles.csv',
                 index_col='show_id')

### Examine data

In [ ]:
df.shape

(9668, 11)

In [ ]:
df.head()

,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
show_id,,,,,,,,,,,
s1,Movie,The Grand Seduction,Don McKellar,"Brendan Gleeson, Taylor Kitsch, Gordon Pinsent",Canada,"March 30, 2021",2014,NaN,113 min,"Comedy, Drama",A small fishing village must procure a local d...
s2,Movie,Take Care Good Night,Girish Joshi,"Mahesh Manjrekar, Abhay Mahajan, Sachin Khedekar",India,"March 30, 2021",2018,13+,110 min,"Drama, International",A Metro Family decides to fight a Cyber Crimin...
s3,Movie,Secrets of Deception,Josh Webber,"Tom Sizemore, Lorenzo Lamas, Robert LaSardo, R...",United States,"March 30, 2021",2017,NaN,74 min,"Action, Drama, Suspense",After a man discovers his wife is cheating on ...
s4,Movie,Pink: Staying True,Sonia Anderson,"Interviews with: Pink, Adele, Beyoncé, Britney...",United States,"March 30, 2021",2014,NaN,69 min,Documentary,"Pink breaks the mold once again, bringing her ..."
s5,Movie,Monster Maker,Giles Foster,"Harry Dean Stanton, Kieran O'Brien, George Cos...",United Kingdom,"March 30, 2021",1989,NaN,45 min,"Drama, Fantasy",Teenage Matt Banting wants to work with a famo...


In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df['description'].head(1)

,description
show_id,
s1,"A small fishing village must procure a local doctor to secure a lucrative business contract. When unlikely candidate and big city doctor Paul Lewis lands in their lap for a trial residence, the townsfolk rally together to charm him into staying. As the doctor's time in the village winds to a close, acting mayor Murray French has no choice but to pull out all the stops."


### Create function to preprocess text

In [ ]:
# Function to clean_text
def clean_text(text):
    lem = WordNetLemmatizer()
    stop = set(stopwords.words('english'))
    punct = string.punctuation
    text = re.sub(r'\s+', ' ', text)
    text = text.translate(str.maketrans('', '', punct)).lower()
    tokens = re.split(r'\W+', text)
    tokens = [lem.lemmatize(word) for word in tokens if word not in stop]
    return ' '.join(tokens)

In [ ]:
%%time

df['description_clean'] = df['description'].apply(clean_text)

CPU times: user 9.35 s, sys: 601 ms, total: 9.95 s
Wall time: 16.3 s


In [ ]:
df[['title', 'description', 'description_clean']].sample(10)

,title,description,description_clean
show_id,,,
s2658,Dino Dana The Movie,"“Dino Dana The Movie” finds 10-year-old Dana, who sees dinosaurs in the real world, completing an experiment that asks where all the kid dinosaurs are. To find the answer, Dana, her older sister Saara, and their new neighbors Mateo and Jadiel go on a dinosaur journey bigger than anything Dana has ever faced before.",dino dana movie find 10yearold dana see dinosaur real world completing experiment asks kid dinosaur find answer dana older sister saara new neighbor mateo jadiel go dinosaur journey bigger anything dana ever faced
s6757,Yakov Smirnoff: Jokes From The Folks,"Experience some of the funniest moments from Yakov's ""Branson Today"" talk show. There's always something funny about Yakov, but there's something even funnier about his audience!",experience funniest moment yakovs branson today talk show there always something funny yakov there something even funnier audience
s6207,The Death Artist,"Walter Paisley, a busboy at a cappuccino bar called the Jabberjaw, is praised as a genius after he kills his landlady's cat and covers it in plaster. Pressured to produce more work, he goes after bigger subjects.",walter paisley busboy cappuccino bar called jabberjaw praised genius kill landlady cat cover plaster pressured produce work go bigger subject
s2327,Go Buster - Educational Cartoons for Kids,"Go Buster is an animated online educational kids series watched all over the globe. Go Buster encourages learning, creativity, and fun! Learn colors, shapes, numbers and more, with our well-loved characters!",go buster animated online educational kid series watched globe go buster encourages learning creativity fun learn color shape number wellloved character
s3888,Deadlocked,"While a zombie virus breaks out, one group of elevator passengers suddenly finds themselves stranded inside while the outbreak ravages the city. This ragtag crew of strangers must band together for a fighting chance of survival against an infected rider and the clever horde that awaits them outside.",zombie virus break one group elevator passenger suddenly find stranded inside outbreak ravage city ragtag crew stranger must band together fighting chance survival infected rider clever horde awaits outside
s2322,"Go with YoYo! Exercise, Yoga and Mindfulness for Kids","GO with YOYO is a place for fitness fun and playful yoga for kids. Jump around and get moving with YOYO, twist into fun yoga poses, and exercise with props you can find at home like your stuffed animal, paper plates and lots more! By blending educational elements with brain boosting movements and our attitude of gratitude, kids are making both their muscles and their mind strong!",go yoyo place fitness fun playful yoga kid jump around get moving yoyo twist fun yoga pose exercise prop find home like stuffed animal paper plate lot blending educational element brain boosting movement attitude gratitude kid making muscle mind strong
s6536,Rising Free,"In 1887 America in the aftermath of racial prejudice, a young woman is on the run for her life through the vast terrain of the Oregon territory. TV-PG-V",1887 america aftermath racial prejudice young woman run life vast terrain oregon territory tvpgv
s2195,HomeMADE,"HomeMADE is the biggest renovation competition ever attempted on television. Ten talented designers from around Australia will compete for a $100,000 cash prize when they take on the biggest challenge of their lives - to completely makeover two suburban family homes every week.",homemade biggest renovation competition ever attempted television ten talented designer around australia compete 100000 cash prize take biggest challenge life completely makeover two suburban family home every week
s732,The Bold and the Beautiful,"The Bold and the Beautiful is a three-time Emmy Award-winning drama set in Los Angeles that tells the story of high fashion glamour, honor, romance, passion and most importantly, family.",bold beautiful t

In [ ]:
# Split data into training and validation sets
df1, df2 = train_test_split(df, test_size=.005, random_state=42)

In [ ]:
df1.shape

(9619, 12)

In [ ]:
df2.shape

(49, 12)

## Part 1
### Shingle is determined by word boundary

### Create function to generate MinHash Forest
* Initialize number of permutations in MinHash
* MinHash the string on all shingles in each document
* Store the MinHash of the string
* Generate a forest of all MinHashed strings
* Index the forest to make it searchable

In [ ]:
def generate_forest(docs, permutations):
    start_time = time.time()

    minhash = []

    for doc in docs:
        m = MinHash(num_perm=permutations)
        for token in doc:                      # Process shingles on word boundary
            m.update(token.encode('utf8'))
        minhash.append(m)

    forest = MinHashLSHForest(num_perm=permutations)

    for i,m in enumerate(minhash):
        forest.add(i,m)

    forest.index()

    print('It took %s seconds to build forest.' %(time.time()-start_time))

    return forest

### Create function to query MinHash Forest
* Preprocess input text into shingles
* Use the same number of permutations for the MinHash as was used to build the forest
* Create a MinHash on the input text using all shingles
* Query the forest with MinHash and return the number of requested recommendations
* Provide the titles of each conference paper recommended

In [ ]:
def predict(text, df, permutations, num_results, forest):
    start_time = time.time()

    m = MinHash(num_perm=permutations)
    for token in text:
        m.update(token.encode('utf8'))

    idx_array = np.array(forest.query(m, num_results))
    if len(idx_array) == 0:
        return None     # if query is empty, return none

    result = df.iloc[idx_array]['title']

    print('It took %s seconds to query forest.' %(time.time()-start_time))

    return result

### Create forest

In [ ]:
# Set number of Permutations
permutations = 128

In [ ]:
forest = generate_forest(df1['description_clean'], permutations)

It took 38.83000659942627 seconds to build forest.


### Make recommendation

In [ ]:
idx = 1
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.005512237548828125 seconds to query forest.

Top 5 recommendations for [Gangstars (Telugu)]:
show_id
s1403             OUT On Stage (The Series)
s484     The Pop Ups: Great Pretenders Club
s4211     Andy Murray: Resurfacing (4K UHD)
s7528                           After Masks
s3425                        A Father's Son
Name: title, dtype: object


In [ ]:
idx = 2
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.003193378448486328 seconds to query forest.

Top 5 recommendations for [Sankarabharanam]:
show_id
s7888          It Had to Be You
s5245                    Sketch
s6758               Underground
s1129    Ross Kemp: Middle East
s301      Tom Walker: Very Very
Name: title, dtype: object


## Part 2
### Shingle size is fixed

In [ ]:
def create_shingles(text, shingle_size=6):
    return [text[i:i+shingle_size] for i in range(len(text)-shingle_size+1)]

In [ ]:
def generate_forest_2(docs, permutations):
    start_time = time.time()

    minhash = []

    for doc in docs:
        shingles = create_shingles(doc)
        m = MinHash(num_perm=permutations)
        for shingle in shingles:
            m.update(shingle.encode('utf8'))
        minhash.append(m)

    forest = MinHashLSHForest(num_perm=permutations)

    for i,m in enumerate(minhash):
        forest.add(i,m)

    forest.index()

    print('It took %s seconds to build forest.' %(time.time()-start_time))

    return forest

In [ ]:
def predict_2(text, df, permutations, num_results, forest):
    start_time = time.time()

    tokens = clean_text(text)
    shingles = create_shingles(tokens)
    m = MinHash(num_perm=permutations)
    for shingle in shingles:
        m.update(shingle.encode('utf8'))

    idx_array = np.array(forest.query(m, num_results))
    if len(idx_array) == 0:
        return None     # if query is empty, return none

    result = df.iloc[idx_array]['title']

    print('It took %s seconds to query forest.' %(time.time()-start_time))

    return result

### Create forest

In [ ]:
forest = generate_forest_2(df1['description_clean'], permutations)

It took 38.221800565719604 seconds to build forest.


### Make recommendations

In [ ]:
idx = 1
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict_2(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.015437126159667969 seconds to query forest.

Top 5 recommendations for [Gangstars (Telugu)]:
show_id
s2391             Gangstars (Hindi)
s4192    Nemr: No Bombing in Beirut
s5185                       Glitch!
s2978               Brash Boys Club
s2390             Gangstars (Tamil)
Name: title, dtype: object


In [ ]:
idx = 2
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict_2(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.008422374725341797 seconds to query forest.

Top 5 recommendations for [Sankarabharanam]:
show_id
s4462                 Parallel
s2036                 Jeremiah
s49      Yoga for Men's Health
s4116        Sugar Valentine 2
s4036                  Archive
Name: title, dtype: object
